# 11. EDA Insights

## Objective

This notebook consolidates the most important findings from the EDA, cleaning, feature engineering, and feature-selection stages.

It is designed to:
- consolidate the key findings from all EDA notebooks
- translate technical observations into business understanding
- identify modeling implications
- define recommendations for the modeling phase


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import pointbiserialr
from sklearn.model_selection import train_test_split

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import RANDOM_STATE, TARGET_COLUMN, TEST_SIZE

raw_data_path = project_root / "data" / "raw" / "telco.csv"
cleaned_data_path = project_root / "data" / "interim" / "telco_churn_cleaned.csv"
engineered_data_path = project_root / "data" / "processed" / "telco_churn_engineered.csv"
feature_selection_notebook_path = project_root / "notebooks" / "10_feature_selection.ipynb"

if not engineered_data_path.exists():
    raise FileNotFoundError(
        "Engineered dataset not found. Run notebook 09_feature_engineering.ipynb first "
        f"to create: {engineered_data_path}"
    )

raw_df = pd.read_csv(raw_data_path) if raw_data_path.exists() else None
cleaned_df = pd.read_csv(cleaned_data_path) if cleaned_data_path.exists() else None
engineered_df = pd.read_csv(engineered_data_path)

df = engineered_df.copy()
if "Churn_bin" not in df.columns:
    raise KeyError("Churn_bin was not found in the engineered dataset.")

target_column = TARGET_COLUMN
id_columns = [column for column in ["customerID"] if column in df.columns]
label_columns = [column for column in ["Churn", "Churn_bin"] if column in df.columns]
feature_columns = [column for column in df.columns if column not in id_columns + label_columns]

X = df[feature_columns].copy()
y = df["Churn_bin"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number", "bool"]).columns.tolist()
binary_features = [
    column
    for column in numeric_features
    if set(pd.Series(X[column]).dropna().astype(float).unique()).issubset({0.0, 1.0})
]
continuous_numeric_features = [column for column in numeric_features if column not in binary_features]
engineered_feature_markers = (
    "_bin",
    "_ordinal",
    "_group",
    "_profile",
    "_log",
    "_band",
    "_x_",
    "is_",
)
engineered_features = [
    column for column in feature_columns if any(marker in column for marker in engineered_feature_markers)
]
original_features = [column for column in feature_columns if column not in engineered_features]

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## Dataset Recap

This section gives a short reminder of the modeling-ready dataset, target definition, feature groups, and current split status.


In [ ]:
dataset_recap = pd.DataFrame(
    {
        "metric": [
            "dataset used for insights",
            "dataset shape",
            "target column",
            "problem type",
            "original feature count",
            "engineered feature count",
            "numeric feature count",
            "categorical feature count",
            "train/test split status",
            "train shape",
            "test shape",
        ],
        "value": [
            str(engineered_data_path.relative_to(project_root)),
            f"{df.shape[0]} rows x {df.shape[1]} columns",
            f"{target_column} / Churn_bin",
            "binary classification",
            len(original_features),
            len(engineered_features),
            len(numeric_features),
            len(categorical_features),
            f"virtual split prepared in notebook using TEST_SIZE={TEST_SIZE}",
            f"{X_train.shape[0]} rows x {X_train.shape[1]} features",
            f"{X_test.shape[0]} rows x {X_test.shape[1]} features",
        ],
    }
)

dataset_recap


## Key Data Quality Findings

This section summarizes the main cleaning findings from earlier notebooks and converts them into a compact action log for modeling.


In [ ]:
quality_rows = []

if raw_df is not None:
    totalcharges_blank_mask = raw_df["TotalCharges"].astype(str).str.strip().eq("")
    quality_rows.append(
        {
            "issue": "Blank TotalCharges values in raw data",
            "finding": int(totalcharges_blank_mask.sum()),
            "detail": "Blank values are present in the raw file and should not be treated as random missingness.",
            "action_taken": "Handled during cleaning and converted into numeric TotalCharges in downstream datasets.",
        }
    )
    if "tenure" in raw_df.columns:
        quality_rows.append(
            {
                "issue": "Blank TotalCharges linked to tenure",
                "finding": int(raw_df.loc[totalcharges_blank_mask, "tenure"].eq(0).sum()),
                "detail": "These records are typically new customers with tenure = 0.",
                "action_taken": "Kept records and cleaned TotalCharges consistently instead of dropping them blindly.",
            }
        )
    quality_rows.append(
        {
            "issue": "Duplicate rows in raw data",
            "finding": int(raw_df.duplicated().sum()),
            "detail": "Checks whether exact duplicates exist in the source dataset.",
            "action_taken": "No large duplicate issue expected for this dataset; continue to keep customerID out of modeling.",
        }
    )

quality_rows.append(
    {
        "issue": "Missing values in engineered dataset",
        "finding": int(df.isna().sum().sum()),
        "detail": "The processed modeling dataset should be numerically stable and ready for downstream pipelines.",
        "action_taken": "Review any remaining nulls before fitting final models.",
    }
)
quality_rows.append(
    {
        "issue": "Duplicate rows in engineered dataset",
        "finding": int(df.duplicated().sum()),
        "detail": "A quick check on the final engineered table.",
        "action_taken": "Duplicate rows should not be passed into modeling if discovered later.",
    }
)

skewness_df = (
    df[continuous_numeric_features]
    .skew(numeric_only=True)
    .sort_values(ascending=False)
    .rename("skewness")
    .reset_index()
    .rename(columns={"index": "feature"})
)
strong_skew = skewness_df.loc[skewness_df["skewness"].abs() >= 1].copy()

quality_rows.append(
    {
        "issue": "Strong skew in continuous numeric variables",
        "finding": int(strong_skew.shape[0]),
        "detail": ", ".join(strong_skew["feature"].head(6).tolist()) if not strong_skew.empty else "No strongly skewed continuous features detected.",
        "action_taken": "Use transformed versions such as TotalCharges_log when they improve stability and interpretability.",
    }
)

quality_summary_df = pd.DataFrame(quality_rows)
quality_summary_df


## Univariate Insights

The goal here is not to recreate every plot, but to extract the most important single-variable patterns that matter for churn understanding and modeling.


In [ ]:
class_balance_df = (
    df["Churn"]
    .value_counts(dropna=False)
    .rename_axis("Churn")
    .reset_index(name="count")
)
class_balance_df["share_pct"] = (class_balance_df["count"] / len(df) * 100).round(2)

numeric_distribution_df = df[continuous_numeric_features].describe().T.reset_index().rename(columns={"index": "feature"})
numeric_distribution_df["skewness"] = numeric_distribution_df["feature"].map(
    dict(zip(skewness_df["feature"], skewness_df["skewness"]))
).round(3)

categorical_cardinality_df = pd.DataFrame(
    {
        "feature": categorical_features,
        "unique_categories": [df[column].nunique(dropna=False) for column in categorical_features],
        "top_category": [df[column].mode(dropna=False).iloc[0] for column in categorical_features],
        "top_category_share_pct": [round(df[column].value_counts(normalize=True, dropna=False).iloc[0] * 100, 2) for column in categorical_features],
    }
).sort_values(["unique_categories", "top_category_share_pct"], ascending=[False, False])

print("Class balance")
display(class_balance_df)
print("\nContinuous feature summary")
display(numeric_distribution_df[["feature", "mean", "std", "min", "25%", "50%", "75%", "max", "skewness"]].sort_values("skewness", ascending=False))
print("\nCategorical dominance snapshot")
display(categorical_cardinality_df.head(12))


**Suggested narrative to write after reviewing the tables:**

- Churn appears moderately imbalanced rather than extremely rare, so evaluation should go beyond raw accuracy.
- Lifecycle and spend variables deserve special attention because their spread and skewness shape how linear models will behave.
- A few categorical groups dominate the dataset, while smaller categories should be watched for sparse one-hot columns during modeling.


## Bivariate Insights With Target

This is one of the most important sections. It summarizes which variables show the clearest relationship with churn and which patterns appear strongest from both statistical and business viewpoints.


In [ ]:
numeric_target_rows = []
for column in continuous_numeric_features + binary_features:
    series = df[column].astype(float)
    corr_value = series.corr(y)
    point_r, point_p = pointbiserialr(y, series)
    numeric_target_rows.append(
        {
            "feature": column,
            "pearson_with_target": round(float(corr_value), 4),
            "abs_pearson_with_target": round(abs(float(corr_value)), 4),
            "pointbiserial_r": round(float(point_r), 4),
            "pointbiserial_p": float(point_p),
        }
    )

numeric_target_df = pd.DataFrame(numeric_target_rows).sort_values(
    "abs_pearson_with_target", ascending=False
).reset_index(drop=True)

categorical_target_rows = []
for column in categorical_features:
    grouped = (
        df.groupby(column, dropna=False)["Churn_bin"]
        .agg(["mean", "count"])
        .reset_index()
        .rename(columns={column: "category_value", "mean": "churn_rate", "count": "segment_count"})
    )
    grouped["feature"] = column
    grouped["overall_gap"] = (grouped["churn_rate"] - df["Churn_bin"].mean()).abs()
    categorical_target_rows.append(grouped)

categorical_target_df = pd.concat(categorical_target_rows, ignore_index=True)
categorical_target_df["churn_rate_pct"] = (categorical_target_df["churn_rate"] * 100).round(2)
categorical_target_df = categorical_target_df.sort_values(
    ["overall_gap", "segment_count"], ascending=[False, False]
).reset_index(drop=True)

print("Top numeric and binary relationships with churn")
display(numeric_target_df.head(15))
print("\nTop categorical segments by churn-rate gap")
display(categorical_target_df[["feature", "category_value", "segment_count", "churn_rate_pct", "overall_gap"]].head(20))


**Suggested narrative to adapt:**

- Contract structure, tenure-related fields, spend-related fields, and support-service variables should show up among the strongest churn signals.
- Payment behavior is often a useful proxy for customer friction, especially when electronic or manual payment groups separate from auto-payment groups.
- Service-support fields such as `OnlineSecurity`, `TechSupport`, and their engineered profiles often reveal practical retention patterns, not just statistical noise.


## Multivariate Insights

This section explains combined patterns instead of isolated variables. It highlights redundancy, interaction structure, and churn-risk profiles that become clearer when variables are read together.


In [ ]:
numeric_corr_matrix = df[continuous_numeric_features + binary_features].corr(numeric_only=True)

high_corr_pairs = []
for left_index, left_feature in enumerate(numeric_corr_matrix.columns):
    for right_feature in numeric_corr_matrix.columns[left_index + 1:]:
        corr_value = numeric_corr_matrix.loc[left_feature, right_feature]
        if abs(corr_value) >= 0.7:
            high_corr_pairs.append(
                {
                    "feature_1": left_feature,
                    "feature_2": right_feature,
                    "correlation": round(float(corr_value), 4),
                    "abs_correlation": round(abs(float(corr_value)), 4),
                }
            )

high_corr_pairs_df = pd.DataFrame(high_corr_pairs).sort_values("abs_correlation", ascending=False)

combined_risk_profiles = (
    df.groupby(["Contract", "payment_method_group", "has_support_services"], dropna=False)["Churn_bin"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "churn_rate", "count": "segment_count"})
)
combined_risk_profiles = combined_risk_profiles.loc[combined_risk_profiles["segment_count"] >= 50].copy()
combined_risk_profiles["churn_rate_pct"] = (combined_risk_profiles["churn_rate"] * 100).round(2)
combined_risk_profiles = combined_risk_profiles.sort_values(["churn_rate", "segment_count"], ascending=[False, False])

print("High-correlation pairs (|r| >= 0.7)")
display(high_corr_pairs_df.head(20) if not high_corr_pairs_df.empty else pd.DataFrame(columns=["feature_1", "feature_2", "correlation", "abs_correlation"]))
print("\nCombined churn-risk profiles")
display(combined_risk_profiles.head(15))


**Suggested narrative to adapt:**

- `TotalCharges`, `tenure`, and spend-derived features are likely to carry overlapping lifecycle information, so they should be reviewed together instead of treated as fully independent predictors.
- Contract, payment, and support combinations usually tell a more actionable story than any single variable alone.
- Engineered interaction fields can be useful, but they should be filtered for redundancy before final linear-model training.


## Feature Engineering Insights

This section summarizes which engineering choices were useful, which ones improved interpretability, and which ones may need restraint during modeling.


In [ ]:
engineering_summary_df = pd.DataFrame(
    {
        "feature_group": [
            "log-transformed fields",
            "binary encodings",
            "ordinal encodings",
            "grouped categorical fields",
            "interaction / profile fields",
            "customer-behavior flags",
        ],
        "matching_features": [
            ", ".join([column for column in engineered_features if column.endswith("_log")]) or "None detected",
            ", ".join([column for column in engineered_features if column.endswith("_bin")]) or "None detected",
            ", ".join([column for column in engineered_features if column.endswith("_ordinal")]) or "None detected",
            ", ".join([column for column in engineered_features if column.endswith("_group") or column.endswith("_band")]) or "None detected",
            ", ".join([column for column in engineered_features if "_profile" in column or "_x_" in column]) or "None detected",
            ", ".join([column for column in engineered_features if column.startswith("is_") or column in {"service_count", "has_support_services", "avg_monthly_spend"}]) or "None detected",
        ],
        "modeling_note": [
            "Use transformed versions when they materially reduce skew or stabilize scale.",
            "Useful for compact linear-model inputs, but avoid keeping raw and encoded duplicates together.",
            "Helpful when there is a natural order, such as contract duration.",
            "Improves interpretability by reducing sparse category fragmentation.",
            "Captures business behavior but can introduce overlapping signal with base columns.",
            "Often strong churn proxies, but should be checked for leakage-style shortcuts and redundancy.",
        ],
    }
)

engineering_summary_df


## Outlier And Scaling Implications

This section clarifies whether observed outliers should be capped, transformed, or left intact, and how scaling should be handled in the modeling phase.


In [ ]:
outlier_rows = []
for column in continuous_numeric_features:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = int(((df[column] < lower_bound) | (df[column] > upper_bound)).sum())
    outlier_rows.append(
        {
            "feature": column,
            "iqr_outlier_count": outlier_count,
            "iqr_outlier_share_pct": round(outlier_count / len(df) * 100, 2),
            "recommended_action": "Review distribution and transform if needed; do not cap automatically if values reflect real customer behavior.",
        }
    )

outlier_summary_df = pd.DataFrame(outlier_rows).sort_values("iqr_outlier_share_pct", ascending=False)

scaling_guidance_df = pd.DataFrame(
    {
        "model_family": [
            "Logistic Regression",
            "Linear SVM / distance-based methods",
            "Tree ensembles (Random Forest, XGBoost, LightGBM)",
        ],
        "scaling_needed": ["Yes", "Yes", "Usually no"],
        "implementation_note": [
            "Apply scaling inside a leakage-safe sklearn pipeline.",
            "Scale numeric features inside the training pipeline only.",
            "Keep preprocessing simpler, but still encode categories consistently.",
        ],
    }
)

print("Outlier summary")
display(outlier_summary_df)
print("\nScaling guidance")
display(scaling_guidance_df)


## Feature Selection Insights

This section bridges notebook 10 into the modeling phase by summarizing the strongest candidate features, redundant families, and feature-handling cautions.


In [ ]:
feature_selection_snapshot = {
    "kept_feature_count": None,
    "review_feature_count": None,
    "dropped_feature_count": None,
    "top_keep_features": [],
}

if feature_selection_notebook_path.exists():
    notebook_json = json.loads(feature_selection_notebook_path.read_text())
    for cell in notebook_json.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source_text = "".join(cell.get("source", []))
        if "final_selection_summary" not in source_text:
            continue
        for output in cell.get("outputs", []):
            output_text = ""
            if "text" in output:
                output_text = "".join(output["text"])
            elif "data" in output and "text/plain" in output["data"]:
                output_text = "".join(output["data"]["text/plain"])
            if "kept_feature_count" in output_text and "top_keep_features" in output_text:
                parsed = json.loads(
                    output_text.replace("'", '"').replace("None", "null")
                )
                feature_selection_snapshot.update(parsed)
                break

feature_selection_summary_df = pd.DataFrame(
    {
        "metric": list(feature_selection_snapshot.keys()),
        "value": [
            ", ".join(value[:15]) if isinstance(value, list) else value
            for value in feature_selection_snapshot.values()
        ],
    }
)

likely_high_value_families_df = pd.DataFrame(
    {
        "feature_family": [
            "contract commitment",
            "customer lifecycle",
            "spend and billing",
            "support and security services",
            "payment behavior",
            "engineered interaction flags",
        ],
        "representative_features": [
            "Contract, Contract_ordinal, is_month_to_month",
            "tenure, tenure_band, is_new_customer",
            "MonthlyCharges, TotalCharges, TotalCharges_log, avg_monthly_spend",
            "OnlineSecurity, TechSupport, has_support_services, profile features",
            "PaymentMethod, payment_method_group, is_auto_payment",
            "tenure_x_MonthlyCharges, contract_payment_profile, internet_*_profile",
        ],
        "modeling_note": [
            "Usually one of the strongest churn families; watch for redundant encodings.",
            "Important for early-churn detection and retention timing.",
            "Potentially strong but partially redundant with tenure-driven lifecycle signal.",
            "Often highly interpretable and useful for targeting retention offers.",
            "Can reflect friction and customer behavior patterns.",
            "Retain selectively; some interaction fields may be too sparse for simple baselines.",
        ],
    }
)

print("Feature-selection snapshot from notebook 10")
display(feature_selection_summary_df)
print("\nFeature families to carry into modeling")
display(likely_high_value_families_df)


## Business Insights

This section should read like an executive-ready summary. The purpose is to convert technical findings into practical churn meaning and retention implications.


In [ ]:
manager_view_df = (
    df.groupby(["Contract", "tenure_band", "has_support_services"], dropna=False)["Churn_bin"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "churn_rate", "count": "segment_count"})
)
manager_view_df = manager_view_df.loc[manager_view_df["segment_count"] >= 40].copy()
manager_view_df["churn_rate_pct"] = (manager_view_df["churn_rate"] * 100).round(2)
manager_view_df = manager_view_df.sort_values(["churn_rate", "segment_count"], ascending=[False, False])

business_takeaways_df = pd.DataFrame(
    {
        "business_theme": [
            "Contract structure",
            "Early lifecycle churn",
            "Value-added support services",
            "Payment behavior",
            "Retention targeting",
        ],
        "implication": [
            "Short-commitment customers usually carry the highest churn exposure.",
            "Newer customers tend to be more fragile and need earlier retention touchpoints.",
            "Security and support-related services may act as stickiness drivers.",
            "Payment patterns can reflect friction, convenience, or customer segment differences.",
            "Retention programs should focus on short-tenure, month-to-month, lower-support segments first.",
        ],
    }
)

print("High-risk business segments")
display(manager_view_df.head(15))
print("\nBusiness interpretation summary")
display(business_takeaways_df)


## Modeling Recommendations

This is the bridge to the next notebook. It should clearly define how the project should move from insights into model training, validation, and comparison.


In [ ]:
modeling_recommendations_df = pd.DataFrame(
    {
        "area": [
            "baseline models",
            "advanced comparisons",
            "validation strategy",
            "class imbalance handling",
            "categorical handling",
            "scaling",
            "evaluation metrics",
            "feature rules",
        ],
        "recommendation": [
            "Start with Logistic Regression as the first benchmark.",
            "Compare with Random Forest, XGBoost, or LightGBM after the baseline is stable.",
            "Use stratified train/test split and stratified cross-validation.",
            "Track class imbalance explicitly; consider class_weight, threshold tuning, and PR-aware evaluation.",
            "Use one-hot encoding for nominal variables inside the pipeline and avoid duplicate raw/encoded representations.",
            "Scale only inside the modeling pipeline for linear or distance-based models.",
            "Monitor ROC-AUC, PR-AUC, F1, recall, precision, and the confusion matrix.",
            "Exclude customerID, review redundant engineered pairs, and guard against leakage-like shortcuts.",
        ],
    }
)

modeling_recommendations_df


## Final EDA Conclusion

The dataset is now in a strong position for modeling. Cleaning decisions have been documented, the main churn drivers are visible, engineered features have improved the signal space, and feature-selection work has narrowed the candidate set into more defensible inputs.

The next phase should move into leakage-safe modeling with a baseline classifier, a small set of stronger comparison models, and evaluation that emphasizes churn-focused metrics rather than raw accuracy alone.
